# 🔢 Token 用量追踪 Demo

用一个简单的 `1+1=?` 例子，演示如何通过 `TokenTracker` 追踪 API 的 token 消耗。

## 1. 基础知识：什么是 Token？

LLM API 按 **token** 计费。Token 不等于字符，大致规则：
- 英文：1 token ≈ 4 个字符 ≈ 0.75 个单词
- 中文：1 token ≈ 1-2 个汉字

每次 API 调用会消耗两部分 token：
- **prompt_tokens**（输入）：你发给模型的 system + user 消息
- **completion_tokens**（输出）：模型生成的回复
- **total_tokens** = prompt_tokens + completion_tokens

我们用 **kTokens（千 token）** 作为单位，方便阅读大数字。比如 45200 tokens = 45.2 kT。

## 2. 先看原始的 API 返回

In [1]:
import os
import sys
from openai import OpenAI
from dotenv import load_dotenv

# 加载 .env 中的 API 密钥
load_dotenv(os.path.join('..', '.env'))

# 初始化客户端
client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    base_url=os.getenv('OPENAI_BASE_URL'),
)

In [2]:
# 发一个最简单的请求：1+1=?
response = client.chat.completions.create(
    model=os.getenv('MODEL', 'Vendor2/Claude-4.6-opus'),
    messages=[{"role": "user", "content": "1+1=?"}],
    max_tokens=50,
)

print('模型回复:', response.choices[0].message.content)

模型回复: 2


In [3]:
# 关键！response 对象自带 usage 属性
print('response.usage 原始对象:')
print(response.usage)
print()
print(f'输入 token 数 (prompt_tokens):     {response.usage.prompt_tokens}')
print(f'输出 token 数 (completion_tokens): {response.usage.completion_tokens}')
print(f'总计 token 数 (total_tokens):      {response.usage.total_tokens}')
print(f'\n换算: {response.usage.total_tokens / 1000:.3f} kTokens')

response.usage 原始对象:
CompletionUsage(completion_tokens=5, prompt_tokens=28, total_tokens=33, completion_tokens_details=None, prompt_tokens_details=None)

输入 token 数 (prompt_tokens):     28
输出 token 数 (completion_tokens): 5
总计 token 数 (total_tokens):      33

换算: 0.033 kTokens


In [11]:
print(response.usage.total_tokens)

33


☝️ 可以看到，即使是 `1+1=?` 这么简单的问题，也消耗了一些 token。

- `prompt_tokens` ≈ 28：包括 user message 的编码开销
- `completion_tokens` ≈ 5：模型回复 "2" 只需要几个 token

## 3. 使用 TokenTracker 追踪用量

我们写了 `scripts/token_tracker.py` 模块来自动化这个过程。核心原理就是：

```
每次 API 调用后 → 从 response.usage 提取 token 数 → 累加记录 → 最后汇总
```

In [4]:
# 导入 TokenTracker
sys.path.insert(0, os.path.join('..', 'scripts'))
from token_tracker import TokenTracker

# 创建 tracker 实例，传入模型名
tracker = TokenTracker(model=os.getenv('MODEL', 'Vendor2/Claude-4.6-opus'))
print('✅ TokenTracker 已创建')

✅ TokenTracker 已创建


### 3.1 模拟多次 API 调用并追踪

In [5]:
# --- 第 1 次调用：简单数学题（模拟 extract 阶段）---
resp1 = client.chat.completions.create(
    model=os.getenv('MODEL', 'Vendor2/Claude-4.6-opus'),
    messages=[{"role": "user", "content": "1+1=?"}],
    max_tokens=50,
)
# 调用后立即记录！
tracker.add(resp1, stage='extract', file='math_test.md')
print(f'调用 1 回复: {resp1.choices[0].message.content}')
print(f'  tokens: {resp1.usage.prompt_tokens} + {resp1.usage.completion_tokens} = {resp1.usage.total_tokens}')

调用 1 回复: 2
  tokens: 28 + 5 = 33


In [6]:
# --- 第 2 次调用：稍复杂的问题（模拟 extract 阶段）---
resp2 = client.chat.completions.create(
    model=os.getenv('MODEL', 'Vendor2/Claude-4.6-opus'),
    messages=[{"role": "user", "content": "What is the capital of France? Answer in one word."}],
    max_tokens=50,
)
tracker.add(resp2, stage='extract', file='geography_test.md')
print(f'调用 2 回复: {resp2.choices[0].message.content}')
print(f'  tokens: {resp2.usage.prompt_tokens} + {resp2.usage.completion_tokens} = {resp2.usage.total_tokens}')

调用 2 回复: Paris
  tokens: 34 + 4 = 38


In [ ]:
# --- 第 3 次调用：验证阶段 ---
resp3 = client.chat.completions.create(
    model=os.getenv('MODEL', 'Vendor2/Claude-4.6-opus'),
    messages=[{"role": "user", "content": "Is 2 the correct answer for 1+1? Reply YES or NO."}],
    max_tokens=50,
)
tracker.add(resp3, stage='verify', file='math_test.md', gene='Addition_Gene')
print(f'调用 3 回复: {resp3.choices[0].message.content}')
print(f'  tokens: {resp3.usage.prompt_tokens} + {resp3.usage.completion_tokens} = {resp3.usage.total_tokens}')

### 3.2 查看汇总

In [7]:
# 打印格式化的汇总表格
tracker.print_summary()


════════════════════════════════════════════════════════════
📊 API Token 用量汇总 (模型: Vendor2/Claude-4.6-opus)
════════════════════════════════════════════════════════════
  阶段              输入 (kT)    输出 (kT)    合计 (kT)     调用次数
  ──────────── ────────── ────────── ────────── ────────
  extract            0.06       0.01       0.07        2
  总计                 0.06       0.01       0.07        2
════════════════════════════════════════════════════════════


In [8]:
# 也可以用代码获取汇总数据
summary = tracker.get_summary()

print('汇总数据（字典格式）:')
import json
print(json.dumps(summary, indent=2, ensure_ascii=False))

汇总数据（字典格式）:
{
  "extract": {
    "prompt_tokens": 62,
    "completion_tokens": 9,
    "total_tokens": 71,
    "prompt_ktokens": 0.06,
    "completion_ktokens": 0.01,
    "total_ktokens": 0.07,
    "calls": 2
  },
  "total": {
    "prompt_tokens": 62,
    "completion_tokens": 9,
    "total_tokens": 71,
    "prompt_ktokens": 0.06,
    "completion_ktokens": 0.01,
    "total_ktokens": 0.07,
    "calls": 2
  }
}


### 3.3 保存用量报告

In [9]:
# 保存为 JSON 报告文件
report_path = tracker.save_report('../reports/token_usage_demo.json')

# 看看报告内容
with open(report_path, 'r') as f:
    report = json.load(f)

print('报告文件内容：')
print(json.dumps(report, indent=2, ensure_ascii=False))

  💾 Token 用量报告已保存: ../reports/token_usage_demo.json
报告文件内容：
{
  "timestamp": "2026-03-13T21:51:13.044230",
  "model": "Vendor2/Claude-4.6-opus",
  "calls": [
    {
      "stage": "extract",
      "file": "math_test.md",
      "prompt_tokens": 28,
      "completion_tokens": 5,
      "total_tokens": 33,
      "timestamp": "2026-03-13T21:41:53.316322"
    },
    {
      "stage": "extract",
      "file": "geography_test.md",
      "prompt_tokens": 34,
      "completion_tokens": 4,
      "total_tokens": 38,
      "timestamp": "2026-03-13T21:43:22.467346"
    }
  ],
  "summary": {
    "extract": {
      "prompt_tokens": 62,
      "completion_tokens": 9,
      "total_tokens": 71,
      "prompt_ktokens": 0.06,
      "completion_ktokens": 0.01,
      "total_ktokens": 0.07,
      "calls": 2
    },
    "total": {
      "prompt_tokens": 62,
      "completion_tokens": 9,
      "total_tokens": 71,
      "prompt_ktokens": 0.06,
      "completion_ktokens": 0.01,
      "total_ktokens": 0.07,
      "cal

## 4. 原理总结

整个追踪流程非常简单，核心就 3 步：

```python
# ① 创建 tracker
tracker = TokenTracker(model='你的模型名')

# ② 每次 API 调用后，把 response 传给 tracker
response = client.chat.completions.create(...)
tracker.add(response, stage='extract', file='xxx.md')
#           ^^^^^^^^
#    tracker 会自动从 response.usage 中提取:
#    - response.usage.prompt_tokens
#    - response.usage.completion_tokens  
#    - response.usage.total_tokens

# ③ 脚本结束时打印汇总
tracker.print_summary()          # 打印表格
tracker.save_report('path.json') # 保存报告
```

在我们的项目中：
- `md_to_json.py`：每处理一个 MD 文件调用一次 API → `tracker.add(response, stage='extract')`
- `verify_response.py`：每验证一个基因调用一次 API → `tracker.add(response, stage='verify')`
- 运行结束后自动输出汇总表格 + 保存 JSON 报告到 `reports/` 目录